In [47]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [48]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
steps = 5000

In [49]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)

In [ ]:
# model declaration
class LongShortTermMemory(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_size
        )
        self.input_linear = nn.Linear(
            in_features=embedding_size, out_features=4 * hidden_size
        )
        self.hidden_linear = nn.Linear(
            in_features=hidden_size, out_features=4 * hidden_size
        )
        self.output_linear = nn.Linear(in_features=hidden_size, out_features=vocab_size)

        # more learning opportunity for forget_gate
        with torch.no_grad():
            self.input_linear.bias[hidden_size : 2 * hidden_size] = 1

    def forward(self, x, state=None):
        batch_size = x.size(0)

        if state is None:
            h = torch.zeros(size=(batch_size, self.hidden_size))
            # long term
            c = torch.zeros(size=(batch_size, self.hidden_size))
        else:
            h, c = state

        logits = []

        for i in range(x.size(1)):
            ch = x[:, i]

            z = self.input_linear(self.embedding(ch)) + self.hidden_linear(h)
            input_gate, forget_gate, data, output_gate = z.chunk(4, dim=-1)

            # how much new information to write
            input_gate = torch.sigmoid(input_gate)
            # how much of the old cell to keep
            forget_gate = torch.sigmoid(forget_gate)
            # the new information itself
            data = torch.tanh(data)
            # how much of the cell to show as h
            output_gate = torch.sigmoid(output_gate)

            c = forget_gate * c + input_gate * data
            h = output_gate * torch.tanh(c)
            logits.append(self.output_linear(h))

        return torch.stack(tensors=logits, dim=1), (h, c)

In [51]:
# model initialization
model = LongShortTermMemory(
    vocab_size=vocab_size, embedding_size=embedding_size, hidden_size=hidden_size
)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 315745


In [52]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch]
    y_batch = y_train[batch]
    train_logits, _ = model(x_batch)
    train_loss = nn.functional.cross_entropy(
        input=train_logits.view(-1, vocab_size), target=y_batch.view(-1)
    )

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits, _ = model(x_val)
            val_loss = nn.functional.cross_entropy(
                input=val_logits.view(-1, vocab_size), target=y_val.view(-1)
            )
            print(
                f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}"
            )

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.182085990905762, val loss = 4.179448127746582
Step 1000: train loss = 1.5426888465881348, val loss = 1.6761775016784668
Step 2000: train loss = 1.4101841449737549, val loss = 1.5956028699874878
Step 3000: train loss = 1.3314251899719238, val loss = 1.5725743770599365
Step 4000: train loss = 1.3374742269515991, val loss = 1.5636730194091797
Step 4999: train loss = 1.3008006811141968, val loss = 1.5595362186431885


In [53]:
# prediction
with torch.inference_mode():
    output = ["R", "O", "M", "E", "O", ":", "\n"]
    max_output_length = 1000
    state = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)

    while len(output) < max_output_length:
        logits, state = model(i, state)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [54]:
print("".join(output))

ROMEO:
But I know no pying to their heart-like;
Your perciful lady's majesty to resemours to extremity.
Thoughts to do the commondern of pradence.

OXFORD:
Adience unluckingham and razils' queens?

Allock:
The life it were I could fear their power;
'Tis not your subject wrong'd
The pruse us lives I am long, thee dead:
Now, how hath sin there not runed itselves!

POLIXENES:
Her doubt?

LADY YORK:
He'll should be food, and I conterponing proud 'vale
Than then they will begett the dangeral this read?

QUEEN ELIZABETH:
But that I make him of those phurse!
But, is, signs our lord, I fight, and with the death-more;
Lend me, swift thee, justic-leaves; in the fury:
Take yout to God protocus, am I sembs to-morrow of the same begin of what,
Your brief, we have not rulies are patience!
Within his may myself
Succent of the thing action and your parl'd
Upon the holy soul did form in hate
Which will put God's royal post;
Now fary a
it for but send to pays into with him;
For comfort thy news? why, th